In [2]:
import time
import math
import os

from tqdm.notebook import tqdm
import numpy as np
import pandas as pd

from sdrsdm import TriadicMemory

import sys

# Сетап

In [35]:
N = 1000
P = 5
SDR_COUNT = 100_000
RNG = np.random.default_rng(100)

In [36]:
def sdr_random():
    inds = RNG.choice(N, P, replace=False)
    inds.sort()
    return inds

def sdr_overlap(s1, s2):
    s1 = set(s1)
    s2 = set(s2)
    return len(s1 & s2)

def sdr_distance(s1, s2):
    return 1 - 2 * overlap(s1, s2) / (len(s1) + len(s2))

In [37]:
mem = TriadicMemory(N, P)

# Генерация векторов

In [38]:
%%time
xx, yy, zz = [], [], []

for _ in tqdm(range(SDR_COUNT)):
    x, y, z = sdr_random(), sdr_random(), sdr_random()
    xx.append(x)
    yy.append(y)
    zz.append(z)

  0%|          | 0/100000 [00:00<?, ?it/s]

CPU times: user 1.61 s, sys: 241 ms, total: 1.85 s
Wall time: 1.52 s


# Сохранение векторов

In [39]:
%%time
for x, y, z in tqdm(zip(xx, yy, zz), total=len(xx)):
    mem.store(x, y, z)

  0%|          | 0/100000 [00:00<?, ?it/s]

CPU times: user 728 ms, sys: 251 ms, total: 980 ms
Wall time: 971 ms


# Тест извлечения векторов

## query_Z

In [40]:
%%time
hits = 0

for x, y, z in tqdm(zip(xx, yy, zz), total=len(xx)):
    z_ret = mem.query_Z(x, y)

    if sdr_distance(z, z_ret) == 0:
        hits += 1

f'{hits} / {len(zz)}'

  0%|          | 0/100000 [00:00<?, ?it/s]

CPU times: user 2.93 s, sys: 12 ms, total: 2.94 s
Wall time: 2.93 s


'100000 / 100000'

## query_X

In [41]:
%%time
hits = 0

for x, y, z in tqdm(zip(xx, yy, zz), total=len(xx)):
    x_ret = mem.query_X(y, z)

    if sdr_distance(x, x_ret) == 0:
        hits += 1

f'{hits} / {len(xx)}'

  0%|          | 0/100000 [00:00<?, ?it/s]

CPU times: user 23 s, sys: 111 ms, total: 23.1 s
Wall time: 23 s


'100000 / 100000'

## query_Y

In [42]:
%%time
hits = 0

for x, y, z in tqdm(zip(xx, yy, zz), total=len(xx)):
    y_ret = mem.query_Y(x, z)

    if sdr_distance(y, y_ret) == 0:
        hits += 1

f'{hits} / {len(yy)}'

  0%|          | 0/100000 [00:00<?, ?it/s]

CPU times: user 17.2 s, sys: 86.2 ms, total: 17.3 s
Wall time: 17.2 s


'100000 / 100000'

# Разница в скорости Z vs X/Y

Разница в скорости между query_Z и query_X/query_Y объясняется большим количеством page_fault при доступе к памяти. Т.к. вектор Z находится последним в размерности куба TriadicMemory, то последовательное вычитывание z компонентов производится из одной/нескольких страниц памяти. Напротив, вычитывание X/Y требует скакания по большому разбросу адресов в памяти с частыми page_fault, необходимостью сбрасывать кеши, подкачивать страницы и т.д.

Код ниже демонстрирует, что просто тупо доступ к памяти X и Z расходится значительно по времени.

In [43]:
L = 1000

In [44]:
%%time
# Scan over x requires big jumps over memory address
for y, z in tqdm(zip(yy[:L], zz), total=L):
    for ay in y:
        for az in z:
            for i in range(mem.mem.shape[0]):
                mem.mem[i, ay, az]

  0%|          | 0/1000 [00:00<?, ?it/s]

CPU times: user 5.65 s, sys: 19.1 ms, total: 5.67 s
Wall time: 5.63 s


In [45]:
%%time
# Scan over z requires access to single (most probale situation) chunk of memory
for x, y in tqdm(zip(xx[:L], yy), total=L):
    for ax in x:
        for ay in y:
            for i in range(mem.mem.shape[-1]):
                mem.mem[ax, ay, i]

  0%|          | 0/1000 [00:00<?, ?it/s]

CPU times: user 3.86 s, sys: 18.1 ms, total: 3.88 s
Wall time: 3.85 s
